# Group Project - Safe routes home at night 

**DS105A Group Project – Data for Data Science (Autumn Term 2025/2026)**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #ED9255; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

**Group Notebook**
- 👤 Team Name: CtrlAltElite
- 🎯 Purpose: This notebook contains our database schema and creates a database.
</div>

In [1]:
import sqlite3
import json
from pathlib import Path
import warnings
import pandas as pd

## Provisional Database Schema

|Table|Columns|Relationships|Each row represents|
|:-----:|:-------:|:-------------:|:-------------------:|
|`areas`|`area_id`(PK), `name`|One-to-many with `clubs`|A distinct London area (e.g. Dalston)|
|`clubs`|`area_id`(FK), `club_id`(PK), `displayName`, `latitude`, `longitude`|One-to-many with `destinations`|A nightclub or bar within a given area designated by `area_id`|
|`destinations`|`club_id`(FK), `destination_id`(PK) `displayName`, `latitude`, `longitude`, `primaryType` |One-to-one with `routes`|A unique nightclub-station pairing|
|`routes`|`destination_id`(FK),`route_id`(PK), `distanceMetres`, `durationSeconds`, |Relationship with crimes expressed in junction table|A unique route from a club to a destination|
|`crimes`|`crime_id`(PK), `category`, `month`, `street_name`, `latitude`, `longitude`|One-to-many with `route_crimes`|A unique crime|
|`route_crimes`|`route_id`(FK), `crime_id`(FK)|Many-to-one with `crimes` and `routes`, respectively|One occurence of a crime falling within one unique route corridor|

## Data Collection Pipeline: 

Step 1: Filter Routes by Night Service Viability
- load the `night_service` JSON file
- filter out routes where `route_viable == False`
- only process viable routes for the database

Step 2: Create SQL Database Schema
- `areas` table (3 rows: Dalston, Hackney_Wick, Bermondsey)
- `clubs` table
- `destinations` table (bus/tube stations)
- `routes` table
- `crimes` table
- `route_crimes` junction table (linking the two tables (routes and crime) with a many-to-many relationship). 

Step 3: Transform and Load Data
- Extract data from JSON files
- Transform to match the schema
- Insert into SQLite database

In [2]:
# Constants
AREA_NAMES = ['Dalston', 'Hackney_Wick', 'Bermondsey']
DB_PATH = 'data/processed/nightclub_safety.db'


In [3]:
def area_slugify(name: str) -> str:
    """Normalize area names for coherent file system."""
    return name.strip().lower().replace(' ', '_').replace('-', '_')

In [4]:
def create_database_schema(conn):
    """
    Create database tables based on provisional schema.
    """
    cursor = conn.cursor()
    
    # Areas table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS areas (
            area_id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL UNIQUE
        )
    ''')
    
    # Clubs table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS clubs (
            club_id INTEGER PRIMARY KEY AUTOINCREMENT,
            area_id INTEGER NOT NULL,
            displayName TEXT NOT NULL,
            latitude REAL NOT NULL,
            longitude REAL NOT NULL,
            FOREIGN KEY (area_id) REFERENCES areas(area_id)
        )
    ''')
    
    # Destinations table (stations)
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS destinations (
            destination_id INTEGER PRIMARY KEY AUTOINCREMENT,
            club_id INTEGER NOT NULL,
            displayName TEXT NOT NULL,
            latitude REAL NOT NULL,
            longitude REAL NOT NULL,
            primaryType TEXT,
            FOREIGN KEY (club_id) REFERENCES clubs(club_id)
        )
    ''')
    
    # Routes table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS routes (
            route_id INTEGER PRIMARY KEY AUTOINCREMENT,
            destination_id INTEGER NOT NULL,
            distanceMeters REAL,
            durationSeconds INTEGER,
            polyline TEXT,
            FOREIGN KEY (destination_id) REFERENCES destinations(destination_id)
        )
    ''')
    
    # Crimes table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS crimes (
            crime_id INTEGER PRIMARY KEY AUTOINCREMENT,
            category TEXT NOT NULL,
            month TEXT NOT NULL,
            street_name TEXT,
            latitude REAL,
            longitude REAL
        )
    ''')
    
    # Junction table for many-to-many relationship
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS route_crimes (
            route_id INTEGER NOT NULL,
            crime_id INTEGER NOT NULL,
            PRIMARY KEY (route_id, crime_id),
            FOREIGN KEY (route_id) REFERENCES routes(route_id),
            FOREIGN KEY (crime_id) REFERENCES crimes(crime_id)
        )
    ''')
    
    conn.commit()
    print("Database schema created successfully")


## 1. Filter Route by Night Service

In [5]:
def check_route_viable(area_slug, club_slug, route_index):
    """
    Check if a route has night service (is viable).
    """
    night_service_path = Path(
        f'data/raw/{area_slug}/night_service/'
        f'{club_slug}_route_{route_index}_night_service.json'
    )
    
    if not night_service_path.exists():
        return False
    
    try:
        with open(night_service_path, 'r') as f:
            night_service = json.load(f)
        return night_service.get('route_viable', False)
    except Exception as e:
        warnings.warn(f"Error reading night service data: {e}")
        return False


## 2. Create SQL Database

In [6]:
def insert_area(conn, area_name):
    """
    Insert area into database and return area_id.
    """
    cursor = conn.cursor()
    
    cursor.execute(
        'INSERT OR IGNORE INTO areas (name) VALUES (?)',
        (area_name,)
    )
    
    cursor.execute('SELECT area_id FROM areas WHERE name = ?', (area_name,))
    area_id = cursor.fetchone()[0]
    
    return area_id


In [7]:
def insert_club(conn, area_id, club_data):
    """
    Insert club into database and return club_id.
    """
    cursor = conn.cursor()
    
    club_name = club_data.get('club', '')
    club_location = club_data.get('club_location', {})
    
    cursor.execute(
        '''INSERT INTO clubs (area_id, displayName, latitude, longitude) 
           VALUES (?, ?, ?, ?)''',
        (area_id, club_name, club_location.get('latitude'), club_location.get('longitude'))
    )
    
    return cursor.lastrowid


In [8]:
def insert_destination(conn, club_id, station_name, station_location, station_type):
    """
    Insert destination (station) into database and return destination_id.
    """
    cursor = conn.cursor()
    
    cursor.execute(
        '''INSERT INTO destinations (club_id, displayName, latitude, longitude, primaryType)
           VALUES (?, ?, ?, ?, ?)''',
        (club_id, station_name, 
         station_location.get('latitude'), 
         station_location.get('longitude'),
         station_type)
    )
    
    return cursor.lastrowid


In [9]:
def insert_route(conn, destination_id, route_data):
    """
    Insert route into database and return route_id.
    """
    cursor = conn.cursor()
    
    # Extract polyline as JSON string
    polyline = route_data.get('polyline', {})
    polyline_json = json.dumps(polyline) if polyline else None
    
    cursor.execute(
        '''INSERT INTO routes (destination_id, distanceMeters, durationSeconds, polyline)
           VALUES (?, ?, ?, ?)''',
        (destination_id,
         route_data.get('distanceMeters'),
         route_data.get('duration'),  # Already in seconds from metadata
         polyline_json)
    )
    
    return cursor.lastrowid


In [10]:
def insert_crime(conn, crime_data):
    """
    Insert crime into database and return crime_id.
    """
    cursor = conn.cursor()
    
    location = crime_data.get('location', {})
    
    cursor.execute(
        '''INSERT INTO crimes (category, month, street_name, latitude, longitude)
           VALUES (?, ?, ?, ?, ?)''',
        (crime_data.get('category'),
         crime_data.get('month'),
         location.get('street', {}).get('name'),
         location.get('latitude'),
         location.get('longitude'))
    )
    
    return cursor.lastrowid


In [11]:
def link_route_crime(conn, route_id, crime_id):
    """
    Create link between route and crime in junction table.
    """
    cursor = conn.cursor()
    
    cursor.execute(
        'INSERT OR IGNORE INTO route_crimes (route_id, crime_id) VALUES (?, ?)',
        (route_id, crime_id)
    )


## 3. Transform Data

In [ ]:
def process_area(conn, area_name):
    """
    Process all data for one area and insert into database using batch operations.
    Only processes routes with night service.
    """
    area_slug = area_slugify(area_name)
    cursor = conn.cursor()
    
    # Insert area
    area_id = insert_area(conn, area_name)
    
    # Get all route files
    routes_dir = Path(f'data/raw/{area_slug}/routes')
    route_files = list(routes_dir.glob('*_routes.json'))
    
    stats = {
        'total_routes': 0,
        'viable_routes': 0,
        'filtered_routes': 0,
        'crimes_inserted': 0
    }
    
    for route_file in route_files:
        club_slug = route_file.stem.replace('_routes', '')
        
        # Load routes data
        with open(route_file, 'r') as f:
            routes_data = json.load(f)
        
        # Insert club
        club_id = insert_club(conn, area_id, routes_data)
        
        # Load metadata directory
        metadata_dir = Path(f'data/raw/{area_slug}/metadata')
        
        # Collect all viable destinations and routes for this club for batch insertion
        destinations_to_insert = []
        routes_to_insert = []
        route_metadata = []  # Store metadata for later crime linking
        
        # Filter and collect data
        for route_index, route in enumerate(routes_data.get('routes', [])):
            stats['total_routes'] += 1
            
            # Check if route is viable (has night service)
            if not check_route_viable(area_slug, club_slug, route_index):
                stats['filtered_routes'] += 1
                continue
            
            stats['viable_routes'] += 1
            
            # Load metadata to get station type
            metadata_path = metadata_dir / f'{club_slug}_route_{route_index}_metadata.json'
            station_type = None
            if metadata_path.exists():
                with open(metadata_path, 'r') as f:
                    metadata = json.load(f)
                    station_type = metadata.get('destination', {}).get('name', '')
            
            # Collect destination data for batch insert
            station_location = route.get('station_location', {})
            destinations_to_insert.append((
                club_id,
                route.get('station_name', ''),
                station_location.get('latitude'),
                station_location.get('longitude'),
                station_type
            ))
            
            # Collect route data for batch insert
            polyline = route.get('polyline', {})
            polyline_json = json.dumps(polyline) if polyline else None
            
            routes_to_insert.append((
                route.get('distanceMeters'),
                route.get('duration'),
                polyline_json
            ))
            
            # Store metadata for crime linking later
            route_metadata.append({
                'area_slug': area_slug,
                'club_slug': club_slug,
                'route_index': route_index
            })
        
        # Batch insert destinations
        if destinations_to_insert:
            cursor.executemany(
                '''INSERT INTO destinations (club_id, displayName, latitude, longitude, primaryType)
                   VALUES (?, ?, ?, ?, ?)''',
                destinations_to_insert
            )
            conn.commit()
            
            # Get the destination IDs that were just inserted by querying back
            cursor.execute(
                '''SELECT destination_id FROM destinations 
                   WHERE club_id = ? 
                   ORDER BY destination_id DESC 
                   LIMIT ?''',
                (club_id, len(destinations_to_insert))
            )
            destination_ids = [row[0] for row in cursor.fetchall()]
            destination_ids.reverse()  # Put back in insertion order
            
            # Prepare routes with their corresponding destination_ids
            routes_with_dest_ids = []
            for i, route_data in enumerate(routes_to_insert):
                # route_data is (distanceMeters, durationSeconds, polyline_json)
                # Add destination_id at the beginning
                routes_with_dest_ids.append(
                    (destination_ids[i], route_data[0], route_data[1], route_data[2])
                )
            
            # Batch insert routes
            cursor.executemany(
                '''INSERT INTO routes (destination_id, distanceMeters, durationSeconds, polyline)
                   VALUES (?, ?, ?, ?)''',
                routes_with_dest_ids
            )
            conn.commit()
            
            # Get the route IDs that were just inserted
            cursor.execute(
                '''SELECT route_id FROM routes 
                   WHERE destination_id IN ({})
                   ORDER BY route_id'''.format(','.join('?' * len(destination_ids))),
                destination_ids
            )
            route_ids = [row[0] for row in cursor.fetchall()]
            
            # Now insert crimes for each route using batch operations
            crime_data_dir = Path(f'data/raw/{area_slug}/crime_data')
            
            for i, metadata in enumerate(route_metadata):
                route_id = route_ids[i]
                
                # Get all crime files for this route
                crime_files = list(crime_data_dir.glob(
                    f'**/{metadata["club_slug"]}_route_{metadata["route_index"]}_crime_data_*.json'
                ))
                
                # Collect all crimes for batch insertion
                crimes_to_insert = []
                
                for crime_file in crime_files:
                    with open(crime_file, 'r') as f:
                        crimes = json.load(f)
                    
                    for crime in crimes:
                        location = crime.get('location', {})
                        crimes_to_insert.append((
                            crime.get('category'),
                            crime.get('month'),
                            location.get('street', {}).get('name'),
                            location.get('latitude'),
                            location.get('longitude')
                        ))
                
                # Batch insert crimes
                if crimes_to_insert:
                    cursor.executemany(
                        '''INSERT INTO crimes (category, month, street_name, latitude, longitude)
                           VALUES (?, ?, ?, ?, ?)''',
                        crimes_to_insert
                    )
                    conn.commit()
                    
                    # Get the crime IDs that were just inserted by querying back
                    # Must match on multiple fields since crimes don't have a unique club/route identifier
                    # Get the most recently inserted crimes (by crime_id) matching our count
                    cursor.execute(
                        '''SELECT crime_id FROM crimes 
                           ORDER BY crime_id DESC 
                           LIMIT ?''',
                        (len(crimes_to_insert),)
                    )
                    crime_ids = [row[0] for row in cursor.fetchall()]
                    crime_ids.reverse()  # Put back in insertion order
                    
                    # Batch insert route-crime links with correct IDs
                    route_crime_links = [
                        (route_id, crime_ids[j]) 
                        for j in range(len(crime_ids))
                    ]
                    
                    cursor.executemany(
                        'INSERT OR IGNORE INTO route_crimes (route_id, crime_id) VALUES (?, ?)',
                        route_crime_links
                    )
                    
                    stats['crimes_inserted'] += len(crimes_to_insert)
    
    conn.commit()
    return stats


In [13]:
def create_database():
    """
    Main function to create and populate the database.
    """
    # Create processed directory
    Path('data/processed').mkdir(parents=True, exist_ok=True)
    
    # Remove existing database if it exists
    if Path(DB_PATH).exists():
        Path(DB_PATH).unlink()
        print(f"Removed existing database at {DB_PATH}")
    
    # Create new database
    conn = sqlite3.connect(DB_PATH)
    print(f"Created new database at {DB_PATH}")
    print()
    
    # Create schema
    create_database_schema(conn)
    print()
    
    # Process each area
    overall_stats = {
        'total_routes': 0,
        'viable_routes': 0,
        'filtered_routes': 0,
        'crimes_inserted': 0
    }
    
    for area_name in AREA_NAMES:
        print(f"Processing {area_name}...")
        stats = process_area(conn, area_name)
        
        overall_stats['total_routes'] += stats['total_routes']
        overall_stats['viable_routes'] += stats['viable_routes']
        overall_stats['filtered_routes'] += stats['filtered_routes']
        overall_stats['crimes_inserted'] += stats['crimes_inserted']
        
        print(f"  Routes: {stats['viable_routes']} viable (filtered {stats['filtered_routes']} without night service)")
        print(f"  Crimes: {stats['crimes_inserted']} inserted")
        print()
    
    # Close connection
    conn.close()
    
    # Print summary
    print("="*70)
    print("Database Creation Complete")
    print("="*70)
    print(f"Total routes collected: {overall_stats['total_routes']}")
    print(f"Viable routes in database: {overall_stats['viable_routes']}")
    print(f"Filtered out (no night service): {overall_stats['filtered_routes']}")
    print(f"Total crimes inserted: {overall_stats['crimes_inserted']}")
    print(f"Database saved to: {DB_PATH}")

# Execute
create_database()

Removed existing database at data/processed/nightclub_safety.db
Created new database at data/processed/nightclub_safety.db

Database schema created successfully

Processing Dalston...
  Routes: 77 viable (filtered 12 without night service)
  Crimes: 2641 inserted

Processing Hackney_Wick...
  Routes: 13 viable (filtered 19 without night service)
  Crimes: 242 inserted

Processing Bermondsey...
  Routes: 75 viable (filtered 11 without night service)
  Crimes: 1434 inserted

Database Creation Complete
Total routes collected: 207
Viable routes in database: 165
Filtered out (no night service): 42
Total crimes inserted: 4317
Database saved to: data/processed/nightclub_safety.db
